In [140]:
from data.LoadData import Data
import numpy as np
import pandas as pd
from typing import Union
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
data = Data(path="../data/car_prices.csv")()

In [59]:
def parse_date(date_str:str):
    try:
        return pd.to_datetime(date_str)
    except :
        return np.nan

def extract_year(date:Union[pd.Timestamp,np.nan]):
    if type(date) is pd.Timestamp:
        return date.year

In [28]:
data.columns

Index(['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state',
       'condition', 'odometer', 'color', 'interior', 'seller', 'mmr',
       'sellingprice', 'saledate'],
      dtype='object')

In [63]:
data.rename(columns={"year":"manufacturing_year"}, inplace=True)

In [145]:
def check_statistical_significance(col:str):
    grouped = data.groupby(by=col)["sellingprice"].apply(list).to_dict()
    return f_oneway(*[group for group in grouped.values() if len(group)>1])

In [146]:
def post_hoc_analysis(grouped:dict):
    all_prices = []
    year_labels = []

    for year, prices in grouped.items():
        all_prices.extend(prices)
        year_labels.extend([year] * len(prices))

    all_prices = np.array(all_prices)
    year_labels = np.array(year_labels)
    tukey_result = pairwise_tukeyhsd(all_prices, year_labels, alpha=0.05)
    print("\nSignificant differences between years:")
    for i, row in enumerate(tukey_result._results_table.data[1:]):
        if row[3] == True:  # If the difference is significant
            print(f"{row[0]} vs {row[1]}: Difference = ${float(row[2]):.2f} (significant)")

In [150]:
for col in data.columns:
    print(f"{col}: {str(data[col].unique())}")

manufacturing_year: [2015 2014 2013 2012 2011 2010 2009 2008 2007 2006 2005 2004 2003 2002
 2001 2000 1999 1998 1995 1996 1997 1987 1994 1993 1992 1989 1991 1990
 1986 1985 1988 1984 1982 1983]
make: ['Kia' 'BMW' 'Volvo' 'Nissan' 'Chevrolet' 'Audi' 'Ford' 'Hyundai' 'Buick'
 'Cadillac' 'Acura' 'Lexus' 'Infiniti' 'Jeep' 'Mercedes-Benz' 'Mitsubishi'
 'Mazda' 'MINI' 'Land Rover' 'Lincoln' 'lincoln' 'Jaguar' 'Volkswagen'
 'Toyota' 'Subaru' 'Scion' 'Porsche' nan 'bmw' 'Dodge' 'FIAT' 'Chrysler'
 'ford' 'Ferrari' 'Honda' 'GMC' 'mitsubishi' 'Ram' 'smart' 'chevrolet'
 'Bentley' 'chrysler' 'pontiac' 'Pontiac' 'Saturn' 'Maserati' 'Mercury'
 'HUMMER' 'landrover' 'cadillac' 'land rover' 'mercedes' 'mazda' 'toyota'
 'lexus' 'gmc truck' 'honda' 'nissan' 'porsche' 'Saab' 'Suzuki' 'dodge'
 'subaru' 'Oldsmobile' 'oldsmobile' 'hyundai' 'jeep' 'Isuzu' 'dodge tk'
 'Geo' 'acura' 'volkswagen' 'suzuki' 'kia' 'audi' 'Rolls-Royce' 'gmc'
 'maserati' 'mazda tk' 'mercury' 'buick' 'hyundai tk' 'mercedes-b' 'vw'
 'Da